# Kimi K3 recreation — train on CodeParrot (Kaggle, 2× T4)

This notebook trains the **Kimi K3 recreation** (Block Attention Residuals + Gated MLA + LatentMoE,
with Gated DeltaNet-2 standing in for Kimi Delta Attention) on the **CodeParrot** Python-code corpus.

**Kaggle** — Settings ▸ Accelerator ▸ **GPU T4 ×2**. Training is data-parallel: parameters are replicated on both T4s and each global batch is split between them (handled inside `train.py`; `--batch-size` is the *global* batch and must be even).

Pipeline (all code lives in the repo, this notebook only drives it):
1. **Grain** — deterministic packed-sequence data loading (`codeparrot_data.py`)
2. **Optax** — Muon + AdamW (`optax.contrib.muon`: Newton-Schulz-orthogonalized momentum for matrix params, per-expert for the MoE stacks; AdamW for embeddings/1-D params), warmup-cosine, grad clip + MoE router-bias balancing (`train.py`)
3. **Orbax** — checkpointing with automatic resume (`train.py` / `evaluate.py`)

> **T4 note:** T4s (Turing) have no native bfloat16, so we keep the default `compute_dtype="float32"`.


## 1. Get the code
Push this project to GitHub first, then set `REPO_URL`.

In [ ]:
import os, pathlib

REPO_URL = "https://github.com/wisnunugroho21/nugie-kimi-k3.git"   # <-- EDIT ME

if not pathlib.Path("nugie-kimi-k3").exists():
    !git clone {REPO_URL} nugie-kimi-k3
%cd nugie-kimi-k3

## 2. Install dependencies
JAX with CUDA wheels, plus Grain / Optax / Orbax and the HF data stack.

In [ ]:
%pip install -q -U "jax[cuda12]" flax grain optax orbax-checkpoint datasets transformers

import os
# Let the Kaggle GPU panel show REAL usage: by default JAX pre-reserves ~75% of
# EVERY visible GPU at startup, so both GPUs always display the same large
# memory footprint whether or not they compute. Disabling preallocation makes
# per-GPU memory reflect actual buffers. (Env vars here are inherited by the
# `!python train.py` subprocess below.)
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

import jax
devices = jax.devices()
print("devices:", devices)
assert len(devices) == 2 and devices[0].platform == "gpu", (
    f"Expected 2 CUDA devices, got {devices}. Fix before training: "
    "Settings -> Accelerator -> GPU T4 x2, then restart the session. "
    "If the accelerator is right, the jax[cuda12] install above failed - "
    "check its output."
)

The cell above **hard-fails unless JAX enumerates 2 CUDA devices** — the one condition data-parallel training needs. If it fails: Settings ▸ Accelerator ▸ GPU T4 ×2, restart, rerun.

## 3. (Optional) Hugging Face token
Add an `HF_TOKEN` secret via Add-ons ▸ Secrets to speed up streaming.

In [ ]:
# Optional: HF token for faster (authenticated) Hub streaming.
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN set")
except Exception:
    print("no HF_TOKEN secret — continuing unauthenticated (fine, just slower)")

## 4. Prepare the CodeParrot data
Streams `codeparrot/codeparrot-clean` (no 50 GB download), tokenizes with the official
`codeparrot/codeparrot` BPE (vocab 32768), packs into `train.bin` / `val.bin` memmaps.
~20M tokens takes a few minutes; raise the budgets for longer runs.

In [ ]:
!python codeparrot_data.py --out data --train-tokens 20000000 --val-tokens 500000

## 5. Train
Re-running this cell (or a fresh session with the same `checkpoints/` dir) **auto-resumes** from the latest Orbax checkpoint — pass a larger `--steps` to extend the run.
The startup log should print `devices: 2 x GPU | per-device batch 32`.

In [ ]:
!python train.py --data-dir data --ckpt-dir checkpoints \
    --steps 3000 --batch-size 64 --seq-len 512 \
    --lr 3e-4 --warmup-steps 200 \
    --log-every 25 --eval-every 500 --eval-batches 20 --ckpt-every 500

### Is the second T4 actually working? Read the training log, not the panel

Kaggle's resource panel is a poor multi-GPU monitor. The trainer now prints the ground
truth **inline in every log line**, sampled from `nvidia-smi` during training:

```
step    25 | loss ... | 61,234 tok/s | gpu 0:72%/3,541MiB 1:70%/3,498MiB
```

* **Both GPUs at similar, nonzero utilization** → data parallelism is working. Done.
* **GPU 1 pinned at 0% across many log lines** → it really is idle; check the startup
  banner: `devices: 2 x GPU ...` and `batch sharded over devices [0, 1]` must both be
  present (if `devices: 1 x GPU` appears, JAX didn't enumerate the second T4 — rerun the
  install cell and check its output).
* **Memory is not evidence either way** — by default JAX reserves ~75% of *both* GPUs at
  startup; this notebook disables that (`XLA_PYTHON_CLIENT_PREALLOCATE=false`) so the
  MiB numbers above are real buffers.
* **Don't expect 2× tokens/s at this model size** — a 49M-param model underutilizes even
  one T4, so per-step host overhead hides much of the speedup. The 2-GPU win grows with
  `--batch-size`, `--seq-len`, and model size.

## 6. Evaluate + sample
Validation cross-entropy / perplexity over the held-out CodeParrot split, then a greedy
code completion through the streaming decode path (GDN-2 recurrent state + MLA latent cache).

In [ ]:
!python evaluate.py --ckpt-dir checkpoints --prompt "def fibonacci(n):" --sample-tokens 64

## 7. Keep the checkpoints
Everything under `/kaggle/working` (including `checkpoints/`) is saved as notebook output
when you **Save Version**. To resume in a later session, attach that output and copy
`checkpoints/` back into the working repo before running the train cell.

## Scaling notes
* `--batch-size` / `--seq-len` — the T4 has plenty of headroom at the default 49M-param model;
  try `--batch-size 64` (Colab) / `128` (Kaggle) if you raise `--steps`.
  `seq_len` must stay a multiple of 64 (the GDN-2 chunk size) and the global batch even.
* Model size — edit the `KimiK3Config` defaults in `kimi_k3_gdn2.py` (`d_model`, `n_layers`,
  `moe_n_routed`, ...). Keep `moe_d_latent ≈ d_model/4` (the LatentMoE recipe).
* Longer data — raise `--train-tokens` in step 4; the packed `.bin` files are reused across runs.